# Salabim Modelling Examples
Replicated from the official docs: https://www.salabim.org/manual/Modelling.html.

Each section mirrors the code listed in the docs. Cells are independent so you can run them separately; note some runs are long or verbose (especially with tracing enabled).

## A simple model (Car)
Minimal component looping with a 1 time-unit hold, demonstrating basic process structure.

In [1]:
# Car.py
import salabim as sim


class Car(sim.Component):
   def process(self):
       while True:
           self.hold(1)


env = sim.Environment(trace=True)
Car()
env.run(till=5)


line#        time current component    action                               information
------ ---------- -------------------- -----------------------------------  ------------------------------------------------
                                       line numbers refers to               2233492516.py
   11                                  default environment initialize       
   11                                  main create                          
   11       0.000 main                 current                              
   12                                  car.0 create                         
   12                                  car.0 activate                       scheduled for 0.000 @    6+ process=process
   13                                  main run +5.000                      scheduled for 5.000 @   13+
    6+      0.000 car.0                current                              
    8                                  car.0 hold +1.000                    scheduled fo

## Bank with 1 clerk (queue + passivate/activate)
Single-clerk FIFO bank with explicit passivation/activation of the clerk and customers.

In [2]:
# Bank, 1 clerk.py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       self.enter(waitingline)
       if clerk.ispassive():
           clerk.activate()
       self.passivate()


class Clerk(sim.Component):
   def process(self):
       while True:
           while len(waitingline) == 0:
               self.passivate()
           self.customer = waitingline.pop()
           self.hold(30)
           self.customer.activate()


env = sim.Environment(trace=True)

CustomerGenerator()
clerk = Clerk()
waitingline = sim.Queue("waitingline")

env.run(till=50)
print()
waitingline.print_statistics()


line#        time current component    action                               information
------ ---------- -------------------- -----------------------------------  ------------------------------------------------
                                       line numbers refers to               2671497579.py
   30                                  default environment initialize       
   30                                  main create                          
   30       0.000 main                 current                              
   32                                  customergenerator.0 create           
   32                                  customergenerator.0 activate         scheduled for 0.000 @    6+ process=process
   33                                  clerk.0 create                       
   33                                  clerk.0 activate                     scheduled for 0.000 @   21+ process=process
   34                                  waitingline create               

## Bank with 3 clerks (passivate/activate)
Three independent clerk components woken up explicitly when work arrives.

In [3]:
# Bank, 3 clerks.py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       self.enter(waitingline)
       for clerk in clerks:
           if clerk.ispassive():
               clerk.activate()
               break  # activate at most one clerk
       self.passivate()


class Clerk(sim.Component):
   def process(self):
       while True:
           while len(waitingline) == 0:
               self.passivate()
           self.customer = waitingline.pop()
           self.hold(30)
           self.customer.activate()


env = sim.Environment(trace=False)
CustomerGenerator()
clerks = [Clerk() for _ in range(3)]

waitingline = sim.Queue("waitingline")

env.run(till=50000)
waitingline.print_histograms()

waitingline.print_info()


Histogram of Length of waitingline
                        all    excl.zero         zero
-------------- ------------ ------------ ------------
duration          50000        48499.381     1500.619
mean                  8.427        8.687
std.deviation         4.852        4.691

minimum               0            1    
median                9           10    
90% percentile       14           14    
95% percentile       16           16    
maximum              21           21    

           <=      duration     %  cum%
        0          1500.619   3.0   3.0 **|                                                                            
        1          2111.284   4.2   7.2 ***  |                                                                         
        2          3528.851   7.1  14.3 *****      |                                                                   
        3          4319.406   8.6  22.9 ******            |                                                       

## Bank with 3 clerks (store)
Uses a Store to buffer customers while clerks pull from it.

In [4]:
# Bank, 3 clerks (store).py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer().enter(waiting_room)
           self.hold(sim.Uniform(5, 15))


class Clerk(sim.Component):
   def process(self):
       while True:
           customer = self.from_store(waiting_room)
           self.hold(30)


class Customer(sim.Component):
   ...


env = sim.Environment(trace=False)
CustomerGenerator()
for _ in range(3):
   Clerk()
waiting_room = sim.Store("waiting_room")


env.run(till=50000)

waiting_room.print_statistics()
waiting_room.print_info()


Statistics of waiting_room at     50000    
                                                                     all    excl.zero         zero
-------------------------------------------- -------------- ------------ ------------ ------------
Length of waiting_room                       duration          50000        48499.381     1500.619
                                             mean                  8.427        8.687
                                             std.deviation         4.852        4.691

                                             minimum               0            1    
                                             median                9           10    
                                             90% percentile       14           14    
                                             95% percentile       16           16    
                                             maximum              21           21    

Length of stay in waiting_room               entries   

## Bank with 3 clerks (resources)
Models clerks as a Resource with capacity 3.

In [5]:
# Bank, 3 clerks (resources).py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       self.request(clerks)
       self.hold(30)
       self.release()  # not really required


env = sim.Environment(trace=False)
CustomerGenerator()
clerks = sim.Resource("clerks", capacity=3)

env.run(till=50000)

clerks.print_statistics()
clerks.print_info()


Statistics of clerks at     50000.000
                                                                     all    excl.zero         zero
-------------------------------------------- -------------- ------------ ------------ ------------
Length of requesters of clerks               duration          50000        48499.381     1500.619
                                             mean                  8.427        8.687
                                             std.deviation         4.852        4.691

                                             minimum               0            1    
                                             median                9           10    
                                             90% percentile       14           14    
                                             95% percentile       16           16    
                                             maximum              21           21    

Length of stay in requesters of clerks       entries         

## Bank with 3 clerks + balking/reneging (passivate/activate)
Adds balking at queue length ≥5 and reneging after waiting 50 time units.

In [6]:
# Example - bank, 3 clerks, reneging.py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       if len(waitingline) >= 5:
           env.number_balked += 1
           env.print_trace("", "", "balked")
           print(env.now(), "balked",self.name())            
           self.cancel()
       self.enter(waitingline)
       for clerk in clerks:
           if clerk.ispassive():
               clerk.activate()
               break  # activate only one clerk
       self.hold(50)  # if not serviced within this time, renege
       if self in waitingline:
           self.leave(waitingline)
           env.number_reneged += 1
           env.print_trace("", "", "reneged")
       else:
           self.passivate()  # wait for service to be completed


class Clerk(sim.Component):
   def process(self):
       while True:
           while len(waitingline) == 0:
               self.passivate()
           self.customer = waitingline.pop()
           self.customer.activate()  # get the customer out of it's hold(50)
           self.hold(30)
           self.customer.activate()  # signal the customer that's all's done


env = sim.Environment()
CustomerGenerator()
env.number_balked = 0
env.number_reneged = 0
clerks = [Clerk() for _ in range(3)]

waitingline = sim.Queue("waitingline")
env.run(duration=300000)
waitingline.length.print_histogram(30, 0, 1)
waitingline.length_of_stay.print_histogram(30, 0, 10)
print("number reneged", env.number_reneged)
print("number balked", env.number_balked)


2027.520282270994 balked customer.206
5321.87964762106 balked customer.534
5364.062889053128 balked customer.539
5603.33045319329 balked customer.564
5800.72378632877 balked customer.586
5841.627163623365 balked customer.591
5888.146653708806 balked customer.597
10663.269633422831 balked customer.1067
14319.716092977507 balked customer.1433
14607.345256580866 balked customer.1463
15018.562522470553 balked customer.1506
15541.849066696866 balked customer.1559
15807.923075156144 balked customer.1587
17829.63397910991 balked customer.1791
17849.813757453063 balked customer.1794
23278.432034335372 balked customer.2337
27328.99525193683 balked customer.2739
27827.6176784221 balked customer.2791
28169.34961331756 balked customer.2826
30755.672671262873 balked customer.3083
32223.384421304185 balked customer.3232
32250.702122569593 balked customer.3236
33892.781587053825 balked customer.3402
34073.453503394485 balked customer.3422
38131.35283677121 balked customer.3825
42059.005328993175 balk

## Bank with 3 clerks + balking/reneging (store)
Implements balking via immediate store fail_at and reneging via a timeout.

In [7]:
# Bank, 3 clerks (store, reneging).py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           customer = Customer()
           self.to_store(waiting_room, customer, fail_at=env.now())
           if self.failed():
               customer.cancel()
               env.number_balked += 1
               print(env.now(), "balked",customer.name())
               env.print_trace("", "", "balked",customer.name())
           self.hold(sim.Uniform(5, 15))


class Clerk(sim.Component):
   def process(self):
       while True:
           customer = self.from_store(waiting_room)
           self.hold(30)


class Customer(sim.Component):
   def process(self):
       self.hold(50)
       if self in waiting_room:
           self.leave(waiting_room)
           env.number_reneged += 1
           env.print_trace("", "", "reneged")

env = sim.Environment(trace=False)
env.number_balked = 0
env.number_reneged = 0
CustomerGenerator()
for _ in range(3):
   Clerk()
waiting_room = sim.Store("waiting_room", capacity=5)

env.run(till=30000)

waiting_room.length.print_histogram(30, 0, 1)
waiting_room.length_of_stay.print_histogram(30, 0, 10)
print("number reneged", env.number_reneged)
print("number balked", env.number_balked)


2027.520282270994 balked customer.206
5321.87964762106 balked customer.534
5364.062889053128 balked customer.539
5603.33045319329 balked customer.564
5800.72378632877 balked customer.586
5841.627163623365 balked customer.591
5888.146653708806 balked customer.597
10663.269633422831 balked customer.1067
14319.716092977507 balked customer.1433
14607.345256580866 balked customer.1463
15018.562522470553 balked customer.1506
15541.849066696866 balked customer.1559
15807.923075156144 balked customer.1587
17829.63397910991 balked customer.1791
17849.813757453063 balked customer.1794
23278.432034335372 balked customer.2337
27328.99525193683 balked customer.2739
27827.6176784221 balked customer.2791
28169.34961331756 balked customer.2826
Histogram of Length of waiting_room
                        all    excl.zero         zero
-------------- ------------ ------------ ------------
duration          30000        24886.090     5113.910
mean                  2.133        2.571
std.deviation         1

## Bank with 3 clerks + balking/reneging (resources)
Uses a Resource with fail_delay for reneging and checks requesters() for balking.

In [8]:
# Example - bank, 3 clerks, reneging (resources).py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       if len(clerks.requesters()) >= 5:
           env.number_balked += 1
           env.print_trace("", "", "balked")
           self.cancel()
       self.request(clerks, fail_delay=50)
       if self.failed():
           env.number_reneged += 1
           env.print_trace("", "", "reneged")
       else:
           self.hold(30)
           self.release()


env = sim.Environment()
CustomerGenerator()
env.number_balked = 0
env.number_reneged = 0
clerks = sim.Resource("clerks", 3)

env.run(till=50000)

clerks.requesters().length.print_histogram(30, 0, 1)
print()
clerks.requesters().length_of_stay.print_histogram(30, 0, 10)
print("number reneged", env.number_reneged)
print("number balked", env.number_balked)


Histogram of Length of requesters of clerks
                        all    excl.zero         zero
-------------- ------------ ------------ ------------
duration          50000        40843.032     9156.968
mean                  2.046        2.504
std.deviation         1.441        1.180

minimum               0            1    
median                2            2    
90% percentile        4            4    
95% percentile        4            4    
maximum               5            5    

           <=      duration     %  cum%
        0          9156.968  18.3  18.3 **************|                                                                
        1         10241.978  20.5  38.8 ****************               |                                               
        2         10947.488  21.9  60.7 *****************                               |                              
        3         10379.802  20.8  81.5 ****************                                                 

## Bank with 3 clerks (state)
Uses a State to signal available work to clerks waiting on it.

In [9]:
# Example - bank, 3 clerks (state).py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       self.enter(waitingline)
       worktodo.trigger(max=1)
       self.passivate()


class Clerk(sim.Component):
   def process(self):
       while True:
           if len(waitingline) == 0:
               self.wait((worktodo, True, 1))
           self.customer = waitingline.pop()
           self.hold(30)
           self.customer.activate()


env = sim.Environment()
CustomerGenerator()
for i in range(3):
   Clerk()
waitingline = sim.Queue("waitingline")
worktodo = sim.State("worktodo")

env.run(till=50000)
waitingline.print_histograms()
worktodo.print_histograms()


Histogram of Length of waitingline
                        all    excl.zero         zero
-------------- ------------ ------------ ------------
duration          50000        48499.381     1500.619
mean                  8.427        8.687
std.deviation         4.852        4.691

minimum               0            1    
median                9           10    
90% percentile       14           14    
95% percentile       16           16    
maximum              21           21    

           <=      duration     %  cum%
        0          1500.619   3.0   3.0 **|                                                                            
        1          2111.284   4.2   7.2 ***  |                                                                         
        2          3528.851   7.1  14.3 *****      |                                                                   
        3          4319.406   8.6  22.9 ******            |                                                       

## Bank with 3 clerks (standby)
Leverages standby() to poll for new customers after every event.

In [10]:
# Example - bank, 3 clerks (standby).py
import salabim as sim


class CustomerGenerator(sim.Component):
   def process(self):
       while True:
           Customer()
           self.hold(sim.Uniform(5, 15).sample())


class Customer(sim.Component):
   def process(self):
       self.enter(waitingline)
       self.passivate()


class Clerk(sim.Component):
   def process(self):
       while True:
           while len(waitingline) == 0:
               self.standby()
           self.customer = waitingline.pop()
           self.hold(30)
           self.customer.activate()


env = sim.Environment(trace=True)
CustomerGenerator()
for _ in range(3):
   Clerk()
waitingline = sim.Queue("waitingline")

env.run(till=50000)
waitingline.length.print_histogram(30, 0, 1)
print()
waitingline.length_of_stay.print_histogram(30, 0, 10)


line#        time current component    action                               information
------ ---------- -------------------- -----------------------------------  ------------------------------------------------
                                       line numbers refers to               854771316.py
   28                                  default environment initialize       
   28                                  main create                          
   28       0.000 main                 current                              
   29                                  customergenerator.0 create           
   29                                  customergenerator.0 activate         scheduled for 0.000 @    6+ process=process
   31                                  clerk.0 create                       
   31                                  clerk.0 activate                     scheduled for 0.000 @   19+ process=process
   31                                  clerk.1 create                    